In [1]:
# Actualizar Jupyter para resolver el problema de IProgress
!pip install --upgrade jupyter

# Reiniciar el kernel después de ejecutar esta celda para aplicar los cambios.

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# Verificar e instalar/actualizar ipywidgets
!pip install --upgrade ipywidgets

# Reiniciar el kernel después de ejecutar esta celda para aplicar los cambios.

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


# Plantilla Base MLOps (MLflow + lakeFS)

Este notebook es una plantilla reutilizable para todos los equipos.

Flujo:
1. Configurar identificadores del caso de uso.
2. Subir dataset a lakeFS y obtener commit hash.
3. Entrenar y registrar ejecuciones en MLflow con trazabilidad.
4. Comparar configuraciones de hiperparametros.

Uso:
- Antes de completar los TODOs, revisar el documento convencion.md:
- Completar los TODOs marcados en el código.
- Ejecutar y verificar en JupyterHub y en la UI de MLflow.

## 1. Imports

In [3]:
import os
from datetime import datetime

import lakefs_sdk
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
import numpy as np
import pandas as pd
from lakefs_sdk.client import LakeFSClient

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

## 2. Configuracion global

Ajustar solo los campos TODO.

Importante:
- Las variables de entorno ya están inyectadas en JupyterHub por lo que no es necesario modificarlas.
- Usar los nombres definidos convencion.md.

In [4]:
import os
import socket
from urllib.parse import urlparse

import mlflow

MLFLOW_DEFAULT_CANDIDATES = [
    os.environ.get("MLFLOW_TRACKING_URI"),
    "http://mlflow:5000",
    "http://simarro-mlflow:5000",
    "http://host.docker.internal:5000",
    "http://localhost:5000",
    "http://127.0.0.1:5000",
]


def _is_reachable(host: str, port: int, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def _pick_reachable_uri(candidates: list[str]) -> tuple[str, list[str]]:
    """Elige la primera URI con host y puerto realmente accesibles por TCP."""
    seen = set()
    checked = []

    for uri in candidates:
        if not uri or uri in seen:
            continue
        seen.add(uri)

        parsed = urlparse(uri)
        host = parsed.hostname
        port = parsed.port or (443 if parsed.scheme == "https" else 80)

        if not host:
            checked.append(f"{uri} -> host invalido")
            continue

        if _is_reachable(host, port):
            checked.append(f"{uri} -> OK")
            return uri, checked

        checked.append(f"{uri} -> sin conexion")

    raise RuntimeError(
        "No se encontro una URI accesible para MLflow. "
        "Configura MLFLOW_TRACKING_URI con una URL alcanzable desde el kernel. "
        f"Intentadas: {checked}"
    )


MLFLOW_URI, _mlflow_checks = _pick_reachable_uri(MLFLOW_DEFAULT_CANDIDATES)
LAKEFS_HOST = os.environ.get("LAKEFS_ENDPOINT", "http://lakefs:8000")
LAKEFS_ACCESS = os.environ.get("LAKEFS_ACCESS_KEY_ID")
LAKEFS_SECRET = os.environ.get("LAKEFS_SECRET_ACCESS_KEY")

START_TIME = datetime.now()

print("Chequeo MLflow:")
for item in _mlflow_checks:
    print(f"  - {item}")

mlflow.set_tracking_uri(MLFLOW_URI)

# Validacion temprana de API de MLflow para fallar con mensaje claro.
try:
    mlflow.tracking.MlflowClient().search_experiments(max_results=1)
    print(f"MLflow OK en: {MLFLOW_URI}")
except Exception as exc:
    raise RuntimeError(
        "Hay conexion TCP pero la API de MLflow no responde correctamente. "
        "Verifica que el servicio este levantado y expuesto en el puerto 5000. "
        f"URI actual: {MLFLOW_URI}"
    ) from exc

# TODO [3/6]: Configurar los identificadores del caso de uso
# Consulta MLOps/compose/convencion.md para los valores correctos.
#
# EXPERIMENT_NAME: nombre del experimento en MLflow
# CASO_USO: letra del caso (B, C, D, E...)
# GRUPO: grupo (G1, G3, G4...)
# DATASET_REPO: repositorio en lakeFS
# REGISTERED_MODEL: nombre oficial en Model Registry
# VARIABLE_TARGET: columna objetivo a predecir
#
# Repositorios por caso (referencia):
#   Caso B: uci-appliances
#   Caso C: lbnl-fdd
#   Caso D: uci-occupancy
#   Caso E: era5
EXPERIMENT_NAME = "CasoD_Prediccion_ocupacion_espacios"
CASO_USO = "D"
GRUPO = "G4"
DATASET_REPO = "casod--uci-occupancy"
REGISTERED_MODEL = "OccupancyClassifier"
VARIABLE_TARGET = "Occupancy"

Chequeo MLflow:
  - http://mlflow:5000/mlflow -> OK
MLflow OK en: http://mlflow:5000/mlflow


## 3. Funciones del pipeline

Incluye subida a lakeFS, carga de datos y entrenamiento con registro en MLflow.

In [5]:
def subir_dataset(ruta_base: str, rama: str = "dev") -> str:
    """
    Sube los 3 archivos del dataset UCI-Occupancy a lakeFS en la rama indicada y hace commit.
    Devuelve el commit hash para usarlo como dataset_version en MLflow.

    Args:
        ruta_base: ruta local a cualquiera de los archivos (se detecta la carpeta base)
        rama: rama de lakeFS donde subir (por defecto 'dev')
    """
    from pathlib import Path

    cfg = lakefs_sdk.Configuration(
        host=LAKEFS_HOST,
        username=LAKEFS_ACCESS,
        password=LAKEFS_SECRET,
    )
    client = LakeFSClient(configuration=cfg)

    # UCI-Occupancy tiene 3 archivos: train + 2 test
    archivos_dataset = ["datatraining.txt", "datatest.txt", "datatest2.txt"]
    base_path = Path(ruta_base).parent

    # Subir cada archivo a lakeFS
    for archivo in archivos_dataset:
        ruta_local = base_path / archivo
        ruta_lakefs = f"data/{archivo}"

        if not ruta_local.exists():
            print(f"Archivo no encontrado: {ruta_local}")
            continue

        with open(ruta_local, "rb") as f:
            content = f.read()
            client.objects_api.upload_object(
                repository=DATASET_REPO,
                branch=rama,
                path=ruta_lakefs,
                content=content,
            )
        print(f"Subido: {archivo}")

    # Hacer commit con todos los archivos.
    # Si lakeFS responde "commit: no changes", reutilizamos el commit actual de la rama.
    try:
        commit = client.commits_api.commit(
            repository=DATASET_REPO,
            branch=rama,
            commit_creation=lakefs_sdk.CommitCreation(
                message=f"feat: dataset {DATASET_REPO} (train + test1 + test2) subido por {GRUPO}",
                metadata={
                    "equipo": GRUPO,
                    "caso_uso": CASO_USO,
                    "archivos": str(len(archivos_dataset)),
                },
            ),
        )
        commit_hash = commit.id
        print("\nDataset UCI-Occupancy subido a lakeFS")
    except lakefs_sdk.exceptions.ApiException as e:
        body = (getattr(e, "body", "") or "").lower()
        if e.status == 400 and "no changes" in body:
            branch_info = client.branches_api.get_branch(repository=DATASET_REPO, branch=rama)
            commit_hash = (
                getattr(branch_info, "commit_id", None)
                or getattr(branch_info, "commit", None)
                or getattr(branch_info, "id", None)
            )
            if not commit_hash:
                raise RuntimeError("No se pudo obtener el commit actual de la rama después de 'no changes'.") from e
            print("\nDataset sin cambios en lakeFS (se reutiliza el ultimo commit de la rama)")
        else:
            raise

    print(f"  Repositorio: {DATASET_REPO}")
    print(f"  Rama: {rama}")
    print(f"  Archivos: {', '.join(archivos_dataset)}")
    print(f"  Commit hash: {commit_hash}")
    return commit_hash


def load_data(ruta_csv: str) -> tuple:
    """
    Cargar el dataset y dividirlo en train/test.

    TODO [5/6]: Adaptar esta funcion al dataset del caso de uso.

    Debes:
      a) Leer CSV con pandas
      b) Separar features (X) y variable objetivo (y)
      c) Eliminar columnas innecesarias (timestamps, IDs, etc.)
      d) Tratar nulos, outliers, etc (fillna, dropna, etc.)

    Return: (X_train, X_test, y_train, y_test)
    """
    from pathlib import Path

    SENSOR_FEATURES = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]

    def _load_uci_file(path: Path) -> pd.DataFrame:
        df = pd.read_csv(path)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
        return df

    base_path = Path(ruta_csv).parent
    train_df = _load_uci_file(base_path / "datatraining.txt")
    test1_df = _load_uci_file(base_path / "datatest.txt")
    test2_df = _load_uci_file(base_path / "datatest2.txt")

    test_df = pd.concat([test1_df, test2_df], ignore_index=True)

    X_train = train_df[SENSOR_FEATURES].copy()
    y_train = train_df[VARIABLE_TARGET].astype(int)

    X_test = test_df[SENSOR_FEATURES].copy()
    y_test = test_df[VARIABLE_TARGET].astype(int)

    X_train = X_train.fillna(X_train.median(numeric_only=True))
    X_test = X_test.fillna(X_train.median(numeric_only=True))

    return X_train, X_test, y_train, y_test


def registrar_report_evidently(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    caso: str,
    dataset: str,
    commit: str,
) -> None:
    """
    Genera un informe básico de drift con Evidently y lo adjunta como artefacto en MLflow
    """
    try:
        from evidently import Report
        from evidently.presets import DataDriftPreset, DataSummaryPreset
        import tempfile
    except Exception as exc:
        print(f"WARN: Evidently no se encuentra disponible: {exc}")
        mlflow.set_tag("evidently_status", "unavailable")
        return

    try:
        current = X_test.copy()
        reference = X_train.copy()

        # Evita errores y problemas con columnas no serializables
        for col in reference.columns:
            if str(reference[col].dtype).startswith("datetime"):
                reference[col] = reference[col].astype(str)
                if col in current.columns:
                    current[col] = current[col].astype(str)

        report = Report(metrics=[DataSummaryPreset(), DataDriftPreset()])
        snapshot = report.run(reference_data=reference, current_data=current)

        with tempfile.TemporaryDirectory(prefix="evidently_") as tmp_dir:
            html_path = os.path.join(tmp_dir, "evidently_report.html")
            json_path = os.path.join(tmp_dir, "evidently_report.json")

            snapshot.save_html(html_path)
            snapshot.save_json(json_path)

            mlflow.log_artifact(html_path, artifact_path="monitoring/evidently")
            mlflow.log_artifact(json_path, artifact_path="monitoring/evidently")

        mlflow.set_tags(
            {
                "evidently_status": "ok",
                "evidently_reference_rows": str(len(reference)),
                "evidently_current_rows": str(len(current)),
                "evidently_case": caso,
                "evidently_dataset": dataset,
                "evidently_dataset_version": commit,
            }
        )
        print("Reporte Evidently registrado en MLflow")

    except Exception as exc:
        print(f"WARN: No se pudo generar report Evidently: {exc}")
        mlflow.set_tag("evidently_status", "error")
        mlflow.set_tag("evidently_error", str(exc)[:250])
        

def train_registry(ruta_csv: str, commit_hash: str, params: dict) -> str:
    """
    Entrena el modelo y registra el run completo en MLflow con trazabilidad.

    Args:
        ruta_csv: ruta local a cualquiera de los archivos UCI
        commit_hash: commit de lakeFS para dataset_version
        params: hiperparámetros del modelo
    """
    # Cargar datos antes de abrir el run para poder entrenar y obtener nombre real del algoritmo
    X_train, X_test, y_train, y_test = load_data(ruta_csv)

    # TODO [6/6]: Cambiar modelo y parámetros según el caso de uso
    # Ejemplos:
    #   Regresion: RandomForestRegressor(**params)
    modelo = RandomForestClassifier(**params)
    modelo.fit(X_train, y_train)

    # Derivar nombre real del algoritmo y construir run_name con timestamp completo
    algoritmo = modelo.__class__.__name__
    timestamp = START_TIME.strftime("%Y%m%d%H%M%S")
    run_name = f"{algoritmo}_{timestamp}_baseline"

    mlflow.set_experiment(EXPERIMENT_NAME)

    client = MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

    created_run = client.create_run(
        experiment_id=experiment.experiment_id,
        start_time=int(START_TIME.timestamp() * 1000),
        tags={
            "mlflow.runName": run_name
        }
    )

    with mlflow.start_run(run_id=created_run.info.run_id) as run:
        run_id = run.info.run_id
        print(f"Run iniciado: {run_id}")
        print(f"Experimento: {EXPERIMENT_NAME}")
        print(f"Nombre run: {run_name}\n")

        # Tags de trazabilidad obligatorios (ver MLOps/compose/convencion.md)
        mlflow.set_tags(
            {
                "caso_uso": CASO_USO,
                "grupo": GRUPO,
                "dataset": DATASET_REPO,
                "dataset_version": commit_hash,
                "dataset_branch": "dev",
                "capa_medallion": "oro",
                "ejecutado_por": os.environ.get("JUPYTERHUB_USER", "local"),
                "algorithm": algoritmo,
            }
        )

        # Registrar hiperparámetros
        mlflow.log_params(params)

        # Registrar metadatos del dataset
        mlflow.set_tags(
            {
                "n_train": len(X_train),
                "n_test": len(X_test),
                "features": ", ".join(X_train.columns.tolist()),
            }
        )

        # Evaluación
        predicciones = modelo.predict(X_test)
        proba = modelo.predict_proba(X_test)
        classes_modelo = np.asarray(modelo.classes_)
        y_test_arr = np.asarray(y_test)
        score_pred = proba.max(axis=1)

        # Ajustar métricas
        # TODO: sustituir por las métricas según el tipo de modelo (ver TODO [2/6])
        accuracy = accuracy_score(y_test, predicciones)
        precision = precision_score(y_test, predicciones, average='weighted', zero_division=0)
        recall = recall_score(y_test, predicciones, average='weighted', zero_division=0)
        f1 = f1_score(y_test, predicciones, average='weighted', zero_division=0)

        mask_validas = np.isin(y_test_arr, classes_modelo)
        if not np.all(mask_validas):
            clases_fuera = np.unique(y_test_arr[~mask_validas]).tolist()
            print(f"Aviso: se ignoran etiquetas no vistas en entrenamiento para AUC: {clases_fuera}")

        y_auc = y_test_arr[mask_validas]
        proba_auc = proba[mask_validas]
        clases_validas = np.array([c for c in classes_modelo if c in np.unique(y_auc)])

        if len(np.unique(y_auc)) < 2:
            auc_roc = None
            print("Aviso: AUC-ROC no disponible porque no hay al menos 2 clases válidas en y_test.")
        elif len(clases_validas) == 2:
            pos_label = clases_validas[1]
            pos_idx = int(np.where(classes_modelo == pos_label)[0][0])
            score_pred = proba[:, pos_idx]
            score_auc = proba_auc[:, pos_idx]
            y_auc_bin = (y_auc == pos_label).astype(int)
            auc_roc = roc_auc_score(y_auc_bin, score_auc)
        else:
            idx_validas = [int(np.where(classes_modelo == c)[0][0]) for c in clases_validas]
            proba_auc_sel = proba_auc[:, idx_validas]
            auc_roc = roc_auc_score(
                y_auc,
                proba_auc_sel,
                labels=clases_validas,
                multi_class='ovr',
                average='weighted',
            )

        metricas = {
            "accuracy": round(accuracy, 4),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
        }
        if auc_roc is not None:
            metricas["auc_roc"] = round(auc_roc, 4)

        mlflow.log_metrics(metricas)

        # Informe de drift de datos
        registrar_report_evidently(
            X_train=X_train,
            X_test=X_test,
            caso=CASO_USO,
            dataset=DATASET_REPO,
            commit=commit_hash,
        )
        
        # Registro del modelo en Model Registry
        model_info = mlflow.sklearn.log_model(
            sk_model=modelo,
            artifact_path="model",
            registered_model_name=REGISTERED_MODEL,
            metadata={
                "caso_uso": CASO_USO,
                "framework": "scikit-learn",
                "task": "classification",
            },
        )

        print(f"Modelo registrado: {REGISTERED_MODEL}")
        print(f"URI: {model_info.model_uri}")

        # Guardar predicciones como artefacto para auditoría
        df_pred = pd.DataFrame(
            {
                "real": y_test.values,
                "prediccion": predicciones,
                "score_modelo": score_pred,
            }
        )
        df_pred.to_csv("/tmp/predicciones.csv", index=False)
        mlflow.log_artifact("/tmp/predicciones.csv", artifact_path="evaluacion")

        return run_id

## 4. Ejecucion del experimento

Actualizar la ruta del CSV y configurar los hiperparámetros del modelo.

Cuando los resultados sean satisfactorios, hacer merge dev --> main para disparar el pipeline automático en la UI de lakeFS.

In [6]:
def ejecutar_experimento():
    # Ajustar esta ruta a la carpeta que contiene los archivos en JupyterHub
    RUTA_BASE = "./datatraining.txt"

    # Paso 1: subir dataset a lakeFS y guardar commit para trazabilidad
    print("=" * 75)
    print("PASO 1: Subiendo dataset a lakeFS...")
    print("=" * 75)
    commit_hash = subir_dataset(RUTA_BASE, rama="dev")

    # TODO: ajusta estas configuraciones al modelo elegido
    configuraciones = [
        {"n_estimators": 100, "max_depth": 6, "random_state": 40, "n_jobs": -1},
        {"n_estimators": 200, "max_depth": 10, "random_state": 40, "n_jobs": -1},
        {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2, "random_state": 40, "n_jobs": -1},
    ]

    # Paso 2: entrenar y registrar cada configuracion en MLflow
    print("\n" + "=" * 75)
    print("PASO 2: Entrenando y registrando en MLflow...")
    print("=" * 75)

    run_ids = []
    for i, params in enumerate(configuraciones, 1):
        print(f"\n[{i}/{len(configuraciones)}] Configuracion: {params}")
        run_id = train_registry(RUTA_BASE, commit_hash, params)
        run_ids.append(run_id)

    print("\n" + "=" * 75)
    print("Experimento completado.")
    print(f"  Revisar en MLflow: {MLFLOW_URI}")
    print(f"  Experimento: {EXPERIMENT_NAME}")
    print("\nSIGUIENTE PASO:")
    print("  Si los resultados son correctos, hacer merge de dev a main en lakeFS")
    print("  para disparar el pipeline de reentrenamiento automático.")
    print("=" * 75)

    return run_ids

In [7]:
# TODO: Descomentar las siguientes líneas cuando estén completados los TODOs
run_ids = ejecutar_experimento()
run_ids

PASO 1: Subiendo dataset a lakeFS...
Subido: datatraining.txt
Subido: datatest.txt
Subido: datatest2.txt

Dataset sin cambios en lakeFS (se reutiliza el ultimo commit de la rama)
  Repositorio: casod--uci-occupancy
  Rama: dev
  Archivos: datatraining.txt, datatest.txt, datatest2.txt
  Commit hash: ff93f4810b788b72fa35a20090ef7ea2dd0169c3da7214c86236c1da333fda0c

PASO 2: Entrenando y registrando en MLflow...

[1/3] Configuracion: {'n_estimators': 100, 'max_depth': 6, 'random_state': 40, 'n_jobs': -1}
Run iniciado: e42467141a8c49a09a785a1a0fbfbf77
Experimento: CasoD_Prediccion_ocupacion_espacios
Nombre run: RandomForestClassifier_20260527083701_baseline



2026/05/27 08:37:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/27 08:37:07 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quie

Reporte Evidently registrado en MLflow


2026/05/27 08:37:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'CasoD_OccupancyClassifier' already exists. Creating a new version of this model...
2026/05/27 08:37:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CasoD_OccupancyClassifier, version 22
Created version '22' of model 'CasoD_OccupancyClassifier'.


Modelo registrado: CasoD_OccupancyClassifier
URI: models:/m-8a0a2234299542e0b1ee59aba904f4d2
🏃 View run RandomForestClassifier_20260527083701_baseline at: http://mlflow:5000/mlflow/#/experiments/3/runs/e42467141a8c49a09a785a1a0fbfbf77
🧪 View experiment at: http://mlflow:5000/mlflow/#/experiments/3

[2/3] Configuracion: {'n_estimators': 200, 'max_depth': 10, 'random_state': 40, 'n_jobs': -1}
Run iniciado: 76ed4ecab8484a8cadb31cdfa9ad601d
Experimento: CasoD_Prediccion_ocupacion_espacios
Nombre run: RandomForestClassifier_20260527083701_baseline



...

Reporte Evidently registrado en MLflow


Registered model 'CasoD_OccupancyClassifier' already exists. Creating a new version of this model...
2026/05/27 08:37:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CasoD_OccupancyClassifier, version 24


Modelo registrado: CasoD_OccupancyClassifier
URI: models:/m-a196a479882b472d9b6cd87ee4009b58
🏃 View run RandomForestClassifier_20260527083701_baseline at: http://mlflow:5000/mlflow/#/experiments/3/runs/ed8aed225cc949f083e03df51bc49c63
🧪 View experiment at: http://mlflow:5000/mlflow/#/experiments/3

Experimento completado.
  Revisar en MLflow: http://mlflow:5000/mlflow
  Experimento: CasoD_Prediccion_ocupacion_espacios

SIGUIENTE PASO:
  Si los resultados son correctos, hacer merge de dev a main en lakeFS
  para disparar el pipeline de reentrenamiento automático.


Created version '24' of model 'CasoD_OccupancyClassifier'.


['e42467141a8c49a09a785a1a0fbfbf77',
 '76ed4ecab8484a8cadb31cdfa9ad601d',
 'ed8aed225cc949f083e03df51bc49c63']